In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Headers of a neural network (from Tutorial of Pytorch Website)

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using xpu device


Use the above to confirm which device is in use in current laptop/desktop, this time it would be XPU

In [5]:
class DemoNeuralNetwork(nn.Module): # initialize the neural network class
    def __init__(self):             # initialize the layers of the neural network through __init__ method
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


Use nn.Flattern() to reduce the dimensions of the data, and then make Pipelines through nn.Sequential(...),
then define forward(self, x) to define the data direction. Here make data flow to x first, and result to the variable logits at last, then return logits.

In [6]:
model = DemoNeuralNetwork().to(device)
print(model)

DemoNeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Use the to(device) and print to make device(XPU) run the network, and print them at last.

In [17]:
X = torch.rand(1, 28, 28, device=torch.device('xpu'))
logits = model(X)                       # logits: raw predictions before applying softmax(results of the last layer)
pred_probab = nn.Softmax(dim=1)(logits) # pred_probab: predicted probabilities for each class after applying softmax
y_pred = pred_probab.argmax(1)          # y_pred: predicted class index with the highest probability
print(f"Predicted class: {y_pred}")

Predicted class: tensor([6], device='xpu:0')


Here use torch.rand() to randomly generate a 28*28 sized pic after each Run. So the consequence can be different at each Run.

In [18]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


Model Layers Breakdown No.1

In [19]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


Use nn.Flatten() to convert the pic into a contiguous array of 784 pixels. The minibatch, which is the dim=0, will be maintained, i.e. the batch number will not be flattened.

In [ ]:
layer1 = nn.Linear(in_features=28*28, out_features=20) # Equivalent to layer1 = nn.Linear(784, 20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


The linear layer is a module that applies a linear transformation on the input using its stored weights and biases. The number of out features will be defined by out_features.

In [23]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1399,  0.5172,  0.0615, -0.3351,  0.3156,  0.7170, -0.6852, -0.0099,
          0.1554, -0.2346,  0.2310, -0.9178,  0.2289,  0.0206, -0.2420, -0.3196,
          0.1348,  0.4088,  0.4619,  0.3751],
        [-0.0831,  0.8275, -0.0372, -0.2079,  0.0734,  0.5197, -0.2650,  0.2013,
          0.5001, -0.5803, -0.1201, -0.5639,  0.0113, -0.0212, -0.2814, -0.1895,
          0.1281,  0.2515,  0.8271,  0.3225],
        [-0.1983,  0.3969, -0.1008, -0.1410, -0.1317,  0.0776, -0.1314, -0.2513,
          0.4722, -0.6326, -0.2320, -0.6122, -0.0118, -0.1903, -0.1472, -0.1965,
         -0.0839,  0.2394,  0.4714,  0.3884]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.5172, 0.0615, 0.0000, 0.3156, 0.7170, 0.0000, 0.0000, 0.1554,
         0.0000, 0.2310, 0.0000, 0.2289, 0.0206, 0.0000, 0.0000, 0.1348, 0.4088,
         0.4619, 0.3751],
        [0.0000, 0.8275, 0.0000, 0.0000, 0.0734, 0.5197, 0.0000, 0.2013, 0.5001,
         0.0000, 0.0000, 0.0000, 0.0113, 0.0000, 0.00

ReLU(x)=max(0,x), this makes the linear data non-linear.

In [24]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: DemoNeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[-0.0200, -0.0170,  0.0078,  ..., -0.0068, -0.0218,  0.0057],
        [ 0.0256,  0.0068,  0.0325,  ..., -0.0251,  0.0093,  0.0166]],
       device='xpu:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0072, -0.0307], device='xpu:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0273, -0.0390, -0.0194,  ...,  0.0160, -0.0239, -0.0257],
        [ 0.0106,  0.0407,  0.0325,  ..., -0.0198, -0.0012, -0.0340]],
       device='xpu:0', grad_fn=<S

In this example, we iterate over each parameter, and print its size and a preview of its values.

In [2]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

loss.backward()
print(w.grad)
print(b.grad)

Gradient function for z = <AddBackward0 object at 0x0000025D26837100>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x0000025D40336650>
tensor([[0.2415, 0.2276, 0.1467],
        [0.2415, 0.2276, 0.1467],
        [0.2415, 0.2276, 0.1467],
        [0.2415, 0.2276, 0.1467],
        [0.2415, 0.2276, 0.1467]])
tensor([0.2415, 0.2276, 0.1467])
